In [1]:
from jussam import load_data
from helpers import *

origin = load_data("insurance_qtcheck")
desc = load_data("insurance_qtcheck_desc")
cat_desc = load_data("insurance_qtcheck_category_desc")

# 명목형 기술통계량 표의 컬럼명이 곧 명목형 변수 목록이다
df = my_qtcheck.set_type(origin, as_category=cat_desc.columns)

c:\Users\itwill\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


📦 연세대학교 주영아 교수가 제작한 라이브러리를 사용중입니다.
📧 Email(1): j.purplerose@yonsei.ac.kr
📧 Email(2): j.purplerose@gmail.com
📝 Website: https://juyounga.kr/
📚 보험 비용 데이터셋의 품질 검사 완료 버전 (출처: 자체 정제)
📚 보험 비용 데이터셋의 연속형 변수에 대한 기술 통계량 (출처: 자체 작업)
📚 보험 비용 데이터셋의 범주형 변수에 대한 기술 통계량 (출처: 자체 작업)
<class 'pandas.DataFrame'>
RangeIndex: 1337 entries, 0 to 1336
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   age       1337 non-null   int64   
 1   sex       1337 non-null   category
 2   bmi       1337 non-null   float64 
 3   children  1337 non-null   int64   
 4   smoker    1337 non-null   category
 5   region    1337 non-null   category
 6   charges   1337 non-null   float64 
dtypes: category(3), float64(2), int64(2)
memory usage: 46.0 KB


In [2]:
target = "charges"

nominal_cols = my_qtcheck.get_categorical_column_names(df)
continuous_cols = my_qtcheck.get_number_column_names(df)
continuous_cols.remove(target)

print("[EDA 반영 전] 명목형 :", nominal_cols)

# EDA에서 채택되지 않은 변수를 걷어낸다
df = df.drop(columns=['sex', 'region'])
cat_desc = cat_desc.drop(columns=['sex', 'region'])
nominal_cols = cat_desc.columns.to_list()

print("[EDA 반영 후] 명목형 :", nominal_cols)
print("           연속형 :", continuous_cols)
print("           크기   :", df.shape)

[EDA 반영 전] 명목형 : ['sex', 'smoker', 'region']
[EDA 반영 후] 명목형 : ['smoker']
           연속형 : ['age', 'bmi', 'children']
           크기   : (1337, 5)


### 1. 명목형 라벨링

In [3]:
df1 = my_prep.labeling(df, columns=nominal_cols)
df1.head()

smoker (2 종): no=0, yes=1


,age,bmi,children,smoker,charges
0,19,27.900,0,1,16884.924
1,18,33.770,1,0,1725.552
2,28,33.000,3,0,4449.462
3,33,22.705,0,0,21984.471
4,32,28.880,0,0,3866.855


### 2. 더미변수 인코딩

In [4]:
df2 = my_prep.dummies(df1, columns=nominal_cols, drop_first=True)
df2.to_excel('insurance_checkpoint_0.xlsx', index=False)
df2.head()

smoker (2종) -> 생략: 이진 변수이므로 원래 컬럼을 유지

컬럼 수: 5개 -> 5개


,age,bmi,children,smoker,charges
0,19,27.900,0,1,16884.924
1,18,33.770,1,0,1725.552
2,28,33.000,3,0,4449.462
3,33,22.705,0,0,21984.471
4,32,28.880,0,0,3866.855


### 3. 로그 변환 대상 컬럼 선정

In [7]:
log_cols = desc.loc[desc['log_need'] == 'log'].index.tolist()
log1p_cols = desc.loc[desc['log_need'] == 'log1p'].index.tolist()
reflect_cols = desc.loc[desc['log_need'] == 'reverse_log1p'].index.tolist()


print('순수log: ', log_cols)
print('log(1+x): ', log1p_cols)
print('반사 후 log: ', reflect_cols)


desc[['skew', 'kurt', 'min', 'log_need']]

순수log:  ['charges']
log(1+x):  ['children']
반사 후 log:  []


,skew,kurt,min,log_need
age,0.055,-1.244,18.000,none
bmi,0.284,-0.053,15.960,none
children,0.937,0.201,0.000,log1p
charges,1.515,1.604,1121.874,log


### 4. 로그 변환

In [8]:
df3 = my_prep.log_transform(df2, log_columns=log_cols,
                            log1p_columns=log1p_cols,
                            reflect_columns=reflect_cols)
df3.to_excel("insurance_checkpoint_1.xlsx", index=False)
df3.head()

컬럼        꼬리방향      변환식                   역변환식                                  왜도
----------------------------------------------------------------------------------------
charges   우측 꼬리     log(x)                exp(y)                   1.51 -> -0.09
children  우측 꼬리     log(1+x)              exp(y)-1                  0.94 -> 0.26


,age,bmi,children,smoker,charges
0,19,27.900,0.000,1,9.734
1,18,33.770,0.693,0,7.453
2,28,33.000,1.386,0,8.401
3,33,22.705,0.000,0,9.998
4,32,28.880,0.000,0,8.260


### 5. 이상치 대체

In [19]:
df4 = my_prep.replace_outlier(df3, columns=continuous_cols)
df4.to_excel("insurance_checkpoint_2.xlsx", index=False)
df4.head()

이상치 대체 방식: 'bound' (기준: IQR x 1.5)
컬럼                               정상 범위       이상치                   대체값
--------------------------------------------------------------------
age                    -9.00 ~ 87.00  0개 ( 0.0%)      -9.00 또는 87.00
bmi                    13.67 ~ 47.32  9개 ( 0.7%)      13.67 또는 47.32
children                -1.65 ~ 2.75  0개 ( 0.0%)       -1.65 또는 2.75


,age,bmi,children,smoker,charges
0,19,27.900,0.000,1,9.734
1,18,33.770,0.693,0,7.453
2,28,33.000,1.386,0,8.401
3,33,22.705,0.000,0,9.998
4,32,28.880,0.000,0,8.260


### 6. 다중공선성 제거

In [14]:
df5 = my_prep.reduce_vif(df4, columns=continuous_cols, threshold=10.0)
df5.to_excel("insurance_checkpoint_3.xlsx", index=False)
df5.head()


완료! 남은 변수: ['age', 'bmi', 'children']
최대 VIF = 1.01


,age,bmi,children,smoker,charges
0,19,27.900,0.000,1,9.734
1,18,33.770,0.693,0,7.453
2,28,33.000,1.386,0,8.401
3,33,22.705,0.000,0,9.998
4,32,28.880,0.000,0,8.260


### 7. 정규화
- 연속형 독립변수를 대상으로 한다
- 정규화는 OLS의 R²를 바꾸지 못한다

In [15]:
df6 = my_prep.scaling(df5, columns=continuous_cols, method='standard')
df6.to_excel('insurance_checkpoint_4.xlsx', index=False)
df6.head()

df6.describe()

StandardScaler 적용 (3개 컬럼)
컬럼                            변환 전                  변환 후
--------------------------------------------------------
age                18.00 ~ 64.00          -1.51 ~ 1.76
bmi                15.96 ~ 47.32          -2.43 ~ 2.75
children             0.00 ~ 1.79          -1.04 ~ 2.17


,age,bmi,children,smoker,charges
count,1337.000,1337.000,1337.000,1337.000,1337.000
mean,-0.000,-0.000,-0.000,0.205,9.100
std,1.000,1.000,1.000,0.404,0.919
min,-1.512,-2.425,-1.043,0.000,7.023
25%,-0.871,-0.720,-1.043,0.000,8.465
50%,-0.016,-0.041,0.199,0.000,9.147
75%,0.839,0.669,0.925,0.000,9.721
max,1.765,2.751,2.167,1.000,11.063
